In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Vision Transformers (ViT) — A Complete Tutorial

> **"An Image is Worth 16×16 Words"** — Dosovitskiy et al., 2020

This notebook is a self-contained lecture + workshop on **Vision Transformers**.
We will:

1. Build a **ViT from scratch** in PyTorch — every module explained step by step
2. Train it on **MNIST**, **Fashion-MNIST**, and **CIFAR-10** locally (runs on MacBook M4 / any CUDA GPU)
3. Visualise **multi-head attention maps** the way the original paper does
4. **Fine-tune a pre-trained ViT** (`google/vit-base-patch16-224`) on a small dataset
5. Run **zero-shot image classification** with **OpenCLIP**
6. Explore large-scale pre-trained encoders: **Google TIPSv2** and **Meta EUPE**

---
## Table of Contents
1. [UV Environment & Installs](#1-uv-environment--installs)
2. [Imports & Device](#2-imports--device)
3. [Directory Layout](#3-directory-layout)
4. [ViT Architecture — every module](#4-vit-architecture--every-module)
5. [Training on MNIST](#5-training-on-mnist)
6. [Training on Fashion-MNIST](#6-training-on-fashion-mnist)
7. [Training on CIFAR-10](#7-training-on-cifar-10)
8. [Attention-Map Visualisation](#8-attention-map-visualisation)
9. [Fine-tuning a Pre-trained ViT (HuggingFace)](#9-fine-tuning-a-pre-trained-vit-huggingface)
10. [Zero-Shot Classification with OpenCLIP](#10-zero-shot-classification-with-openclip)
11. [TIPSv2 & EUPE — Large-Scale ViT Feature Visualisation](#11-tipsv2--eupe--large-scale-vit-feature-visualisation)

## 1. UV Environment & Installs

This notebook is intended to run inside a **[uv](https://github.com/astral-sh/uv)** managed
Python environment — the fastest way to get reproducible installs on macOS (Apple Silicon)
and Linux.

### First-time setup (run once in your terminal)

```bash
# 1. Install uv if you haven't already
curl -LsSf https://astral.sh/uv/install.sh | sh

# 2. Create a project venv and install all dependencies
uv venv .venv --python 3.11
source .venv/bin/activate

# 3. Install PyTorch (MPS-enabled on Apple Silicon)
uv pip install torch torchvision torchaudio

# 4. Install notebook extras
uv pip install jupyter ipykernel tqdm matplotlib pillow

# 5. Install HuggingFace / CLIP extras
uv pip install transformers datasets open_clip_torch sentencepiece scikit-learn

# 6. Register the kernel
python -m ipykernel install --user --name vit-tutorial --display-name "Python (vit-tutorial)"
```

Then launch Jupyter and select the **"Python (vit-tutorial)"** kernel.

In [ ]:
# Run this once if any import below fails.
# sentencepiece is required for TIPSv2 text tokenisation.
# !uv pip install -q tqdm transformers datasets open_clip_torch sentencepiece scikit-learn timm huggingface_hub

## 2. Imports & Device

In [ ]:
import os, math, warnings, subprocess, sys
from pathlib import Path
from collections import defaultdict
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as T
from torchvision.datasets import MNIST, FashionMNIST, CIFAR10

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

### Device Detection

The notebook auto-detects **CUDA** (NVIDIA GPU), **MPS** (Apple Silicon M-series),
or falls back to **CPU**.

In [ ]:
def get_device() -> torch.device:
    return torch.device("cuda") if torch.cuda.is_available() else (
        torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
    )

DEVICE = get_device()
print(f"Using device: {DEVICE}")

## 3. Directory Layout

We use a predictable folder structure so every training run is reproducible
and checkpoints are easy to find:

```
data/
├── vit/                   ← trained model weights (.pth files, one per dataset)
│   ├── mnist_vit.pth
│   ├── fashion_mnist_vit.pth
│   ├── cifar10_vit.pth
│   └── cifar10_vit_finetuned.pth
├── MNIST/                 ← auto-downloaded by torchvision
├── FashionMNIST/
└── cifar-10-batches-py/

models/
└── vit/                   ← HuggingFace model cache (optional)
```

**Training is skipped automatically** if a checkpoint already exists,
so you can re-run cells without waiting for training again.

In [ ]:
DATA_DIR  = Path("data")
VIT_DIR   = DATA_DIR / "vit"       # ← all trained weights live here
MODEL_DIR = Path("models") / "vit"  # optional HF cache

for d in [DATA_DIR, VIT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories ready:")
print(f"  {DATA_DIR.resolve()}")
print(f"  {VIT_DIR.resolve()}")
print(f"  {MODEL_DIR.resolve()}")

## 4. ViT Architecture — Every Module

### 4.1 The Big Picture

The original **ViT** (Dosovitskiy et al., 2020) adapts the Transformer architecture —
originally designed for NLP — to work on images. The key insight is beautifully simple:

> **Split the image into fixed-size patches, flatten each patch into a vector,
> treat the sequence of vectors exactly like a sequence of word tokens.**

```
 ┌──────────────────────────────────────────────────────┐
 │  INPUT IMAGE  (H × W × C)                           │
 └──────────────────┬───────────────────────────────────┘
                    │  Split into N patches (P × P × C each)
                    ▼
 ┌──────────────────────────────────────────────────────┐
 │  PATCH EMBEDDING  (N × D)   +  [CLS] token           │
 └──────────────────┬───────────────────────────────────┘
                    │  + Positional Encoding
                    ▼
 ┌──────────────────────────────────────────────────────┐
 │  TRANSFORMER ENCODER  ×L layers                      │
 │    each layer:  LayerNorm → MSA → residual           │
 │                 LayerNorm → MLP → residual           │
 └──────────────────┬───────────────────────────────────┘
                    │  Take [CLS] representation
                    ▼
 ┌──────────────────────────────────────────────────────┐
 │  MLP HEAD  →  class logits                           │
 └──────────────────────────────────────────────────────┘
```

Number of patches:  `N = (H/P) × (W/P)`,  sequence length = `N + 1`  (the +1 is `[CLS]`).

**Why a `[CLS]` token?**  Just like BERT — it acts as a global summary of the whole sequence
and is used for the final classification.  You could also average-pool all patch tokens
(called *GAP ViT*), and it often works equally well.

### 4.2 Patch Embedding

The simplest way to embed patches is a **single `Conv2d`** with kernel size = stride = patch size.
For a `(H, W, C)` image and patch size `P`, this produces a feature map of shape
`(D, H/P, W/P)` — equivalently, `N = (H/P)*(W/P)` vectors of dimension `D`.

This is identical to extracting every patch, flattening it, and multiplying by a weight matrix,
but the `Conv2d` implementation is faster and handles the extraction automatically.

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Converts an image into a sequence of patch embeddings.

    Args:
        image_size:  side length of (square) input image
        patch_size:  side length of each (square) patch
        in_channels: number of input channels (1 for grayscale, 3 for RGB)
        embed_dim:   output embedding dimension D
    """

    def __init__(self, image_size: int, patch_size: int,
                 in_channels: int, embed_dim: int):
        super().__init__()
        assert image_size % patch_size == 0, "Image size must be divisible by patch size"
        self.num_patches = (image_size // patch_size) ** 2
        # Single Conv2d does patch extraction + linear projection in one step
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x: Tensor) -> Tensor:
        # x: (B, C, H, W)
        x = self.proj(x)          # (B, D, H/P, W/P)
        x = x.flatten(2)          # (B, D, N)
        x = x.transpose(1, 2)     # (B, N, D)
        return x

# Smoke test
_pe  = PatchEmbedding(image_size=28, patch_size=7, in_channels=1, embed_dim=64)
_img = torch.zeros(2, 1, 28, 28)
print("Patch embedding output:", _pe(_img).shape)  # (2, 16, 64)

### 4.3 Multi-Head Self-Attention (MSA)

Self-attention lets every patch attend to every other patch — giving ViT its global receptive
field from the very first layer (unlike CNNs which build it layer by layer).

For each head `h` out of `H` total heads, we compute:

$$\text{head}_h = \text{Softmax}\!\left(\frac{Q_h K_h^\top}{\sqrt{d_k}}\right) V_h$$

$$\text{MSA}(x) = \text{concat}(\text{head}_1, \ldots, \text{head}_H)\,W^O$$

where `d_k = D / H` is the per-head dimension.

The `attention_weights` tensor — shape `(B, H, N+1, N+1)` — is what we will visualise later.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """
    Scaled dot-product multi-head self-attention.

    Args:
        embed_dim:   total embedding dimension D
        num_heads:   number of attention heads H  (D must be divisible by H)
        attn_drop:   dropout on attention weights
        proj_drop:   dropout after output projection
    """

    def __init__(self, embed_dim: int, num_heads: int,
                 attn_drop: float = 0.0, proj_drop: float = 0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5

        self.qkv  = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj_drop = nn.Dropout(proj_drop)

        # Saved during forward for visualisation
        self.last_attn_weights: Optional[Tensor] = None

    def forward(self, x: Tensor) -> Tensor:
        B, N, D = x.shape
        H, d = self.num_heads, self.head_dim

        # (B, N, 3D) → (3, B, H, N, d)
        qkv = self.qkv(x).reshape(B, N, 3, H, d).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)                         # each: (B, H, N, d)

        attn = (q @ k.transpose(-2, -1)) * self.scale   # (B, H, N, N)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        self.last_attn_weights = attn.detach()           # save for visualisation

        x = (attn @ v).transpose(1, 2).reshape(B, N, D)
        return self.proj_drop(self.proj(x))

### 4.4 MLP Block

After attention, each position is processed independently by a small **two-layer MLP**
with a GELU activation.  The hidden dimension is typically `4 × D` (controlled by `mlp_ratio`).

In [ ]:
class MLPBlock(nn.Module):
    """Position-wise feed-forward network inside each Transformer block."""

    def __init__(self, embed_dim: int, mlp_ratio: float = 4.0, drop: float = 0.0):
        super().__init__()
        hidden = int(embed_dim * mlp_ratio)
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden, embed_dim),
            nn.Dropout(drop),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

### 4.5 Transformer Encoder Block

One encoder block = **LayerNorm → MSA → residual** then **LayerNorm → MLP → residual**.

Pre-norm (norm *before* the sub-layer) is the standard ViT choice — it is more stable
to train than the original post-norm Transformer.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """Single Transformer encoder layer (pre-norm variant)."""

    def __init__(self, embed_dim: int, num_heads: int,
                 mlp_ratio: float = 4.0,
                 attn_drop: float = 0.0, drop: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = MultiHeadSelfAttention(embed_dim, num_heads, attn_drop, drop)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp   = MLPBlock(embed_dim, mlp_ratio, drop)

    def forward(self, x: Tensor) -> Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

### 4.6 Full Vision Transformer

Putting it all together:

1. **PatchEmbedding** converts the image to `(B, N, D)`
2. A learnable `[CLS]` token is prepended → `(B, N+1, D)`
3. Learnable **positional embeddings** are added (the model learns spatial structure)
4. `L` **TransformerEncoderBlocks** process the sequence
5. The `[CLS]` token's output is passed through a **classification head**

> **Positional encoding is crucial** — without it, ViT is a *bag of patches* and has no idea
> about spatial layout.  Unlike NLP which often uses sinusoidal encodings, ViT uses learnable
> 1D position embeddings; experiments in the paper showed this works just as well.

In [ ]:
class VisionTransformer(nn.Module):
    """
    Minimal Vision Transformer for image classification.

    Args:
        image_size:  H = W of input image
        patch_size:  size of each patch
        in_channels: 1 (grayscale) or 3 (RGB)
        num_classes: output classes
        embed_dim:   token dimension D
        num_heads:   attention heads per layer
        num_layers:  number of Transformer encoder blocks
        mlp_ratio:   MLP hidden dim = mlp_ratio × embed_dim
        attn_drop:   attention dropout rate
        drop_rate:   general dropout rate
    """

    def __init__(
        self,
        image_size: int   = 28,
        patch_size: int   = 7,
        in_channels: int  = 1,
        num_classes: int  = 10,
        embed_dim: int    = 64,
        num_heads: int    = 4,
        num_layers: int   = 4,
        mlp_ratio: float  = 4.0,
        attn_drop: float  = 0.0,
        drop_rate: float  = 0.0,
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(image_size, patch_size, in_channels, embed_dim)
        N = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, N + 1, embed_dim))
        self.pos_drop  = nn.Dropout(drop_rate)

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, attn_drop, drop_rate)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: Tensor) -> Tensor:
        B = x.shape[0]
        x   = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = self.pos_drop(x + self.pos_embed)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])

    def get_attention_weights(self, layer: int = -1) -> Optional[Tensor]:
        """Return attention weights saved during the last forward pass."""
        return self.blocks[layer].attn.last_attn_weights

# Smoke test
_vit = VisionTransformer()
_x   = torch.zeros(2, 1, 28, 28)
print("ViT output shape:", _vit(_x).shape)
print(f"Total parameters: {sum(p.numel() for p in _vit.parameters()):,}")

### 4.7 Training & Evaluation Utilities

All training loops use **`tqdm`** progress bars with live loss/accuracy readouts.
Each loop shows:
- An outer epoch bar with current validation accuracy
- An inner batch bar with the current batch loss

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc="  batch", leave=False, unit="batch")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
        pbar.set_postfix(loss=f"{loss.item():.3f}", acc=f"{correct/total:.3f}")
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        correct += (model(imgs).argmax(1) == labels).sum().item()
        total   += imgs.size(0)
    return correct / total


def train(model, train_loader, val_loader, epochs, lr, device, label="Training"):
    """Full training loop with cosine LR and tqdm progress bars."""
    optimizer   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    total_steps = epochs * len(train_loader)
    scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

    history   = defaultdict(list)
    epoch_bar = tqdm(range(1, epochs + 1), desc=label, unit="epoch")
    for epoch in epoch_bar:
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        val_acc = evaluate(model, val_loader, device)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(val_acc)
        epoch_bar.set_postfix(
            loss=f"{tr_loss:.4f}", tr=f"{tr_acc:.3f}", val=f"{val_acc:.3f}")
    return history


def plot_history(histories: dict, title="Training History"):
    """Plot training curves for one or more runs."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for name, h in histories.items():
        axes[0].plot(h["train_loss"], label=name)
        axes[1].plot(h["val_acc"],    label=name)
    axes[0].set_title("Train Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].set_title("Val Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()

## 5. Training on MNIST

MNIST images are **28 × 28 grayscale**.  We use `patch_size=7`, giving `4 × 4 = 16` patches.

| Hyper-parameter | Value |
|---|---|
| image size | 28 × 28 |
| patch size | 7 × 7 |
| num patches | 16 |
| embed dim | 128 |
| heads | 4 |
| layers | 4 |
| parameters | ~500K |

**Checkpoint caching:** if `data/vit/mnist_vit.pth` exists the cell loads it and
skips training — useful when iterating on later sections.

In [ ]:
MNIST_MEAN, MNIST_STD = (0.1307,), (0.3081,)

mnist_train_tf = T.Compose([T.RandomAffine(degrees=10, translate=(0.1, 0.1)),
                             T.ToTensor(),
                             T.Normalize(MNIST_MEAN, MNIST_STD)])
mnist_test_tf  = T.Compose([T.ToTensor(), T.Normalize(MNIST_MEAN, MNIST_STD)])

mnist_train = MNIST(DATA_DIR, train=True,  download=True, transform=mnist_train_tf)
mnist_test  = MNIST(DATA_DIR, train=False, download=True, transform=mnist_test_tf)

mnist_train_loader = DataLoader(mnist_train, batch_size=256, shuffle=True,  num_workers=0)
mnist_test_loader  = DataLoader(mnist_test,  batch_size=512, shuffle=False, num_workers=0)
print(f"MNIST  train: {len(mnist_train):,}  |  test: {len(mnist_test):,}")

# Visualise a few samples
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flatten()):
    img, label = mnist_train.data[i], mnist_train.targets[i]
    ax.imshow(img, cmap="gray")
    ax.set_title(str(label.item()), fontsize=9)
    ax.axis("off")
plt.suptitle("MNIST samples", y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
MNIST_CKPT = VIT_DIR / "mnist_vit.pth"

mnist_vit = VisionTransformer(
    image_size=28, patch_size=7, in_channels=1, num_classes=10,
    embed_dim=128, num_heads=4, num_layers=4,
    mlp_ratio=4.0, attn_drop=0.1, drop_rate=0.1,
).to(DEVICE)

if MNIST_CKPT.exists():
    mnist_vit.load_state_dict(torch.load(MNIST_CKPT, map_location=DEVICE, weights_only=True))
    print(f"Loaded checkpoint: {MNIST_CKPT}")
    mnist_history = None
else:
    n = sum(p.numel() for p in mnist_vit.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n:,}")
    mnist_history = train(
        model=mnist_vit, train_loader=mnist_train_loader,
        val_loader=mnist_test_loader, epochs=25, lr=3e-4,
        device=DEVICE, label="MNIST ViT",
    )
    torch.save(mnist_vit.state_dict(), MNIST_CKPT)
    print(f"\nCheckpoint saved → {MNIST_CKPT}")

final_mnist_acc = evaluate(mnist_vit, mnist_test_loader, DEVICE)
print(f"MNIST test accuracy: {final_mnist_acc:.4f}  ({final_mnist_acc*100:.2f}%)")

if mnist_history:
    plot_history({"MNIST ViT": mnist_history}, title="ViT on MNIST")

## 6. Training on Fashion-MNIST

**Fashion-MNIST** has the same 28 × 28 shape but replaces digits with 10 clothing categories.
It is a harder dataset — a simple CNN baseline gets ~92 %, state-of-the-art is ~96 %.

We reuse the exact same architecture to show that ViT generalises well across different
visual domains without any architecture changes.

**What to observe:**
- Similar convergence speed to MNIST
- Lower absolute accuracy (harder task)
- Attention maps will focus on the *shape* of clothing items

In [ ]:
FM_MEAN, FM_STD = (0.2860,), (0.3530,)

fm_train_tf = T.Compose([T.RandomHorizontalFlip(),
                          T.RandomAffine(degrees=10, translate=(0.1, 0.1)),
                          T.ToTensor(),
                          T.Normalize(FM_MEAN, FM_STD)])
fm_test_tf  = T.Compose([T.ToTensor(), T.Normalize(FM_MEAN, FM_STD)])

fm_train = FashionMNIST(DATA_DIR, train=True,  download=True, transform=fm_train_tf)
fm_test  = FashionMNIST(DATA_DIR, train=False, download=True, transform=fm_test_tf)

fm_train_loader = DataLoader(fm_train, batch_size=256, shuffle=True,  num_workers=0)
fm_test_loader  = DataLoader(fm_test,  batch_size=512, shuffle=False, num_workers=0)

FM_CLASSES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
              "Sandal",  "Shirt",   "Sneaker",  "Bag",   "Ankle boot"]

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flatten()):
    img, label = fm_train.data[i], fm_train.targets[i]
    ax.imshow(img, cmap="gray")
    ax.set_title(FM_CLASSES[label.item()], fontsize=7)
    ax.axis("off")
plt.suptitle("Fashion-MNIST samples", y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
FM_CKPT = VIT_DIR / "fashion_mnist_vit.pth"

fm_vit = VisionTransformer(
    image_size=28, patch_size=7, in_channels=1, num_classes=10,
    embed_dim=128, num_heads=4, num_layers=4,
    mlp_ratio=4.0, attn_drop=0.1, drop_rate=0.1,
).to(DEVICE)

if FM_CKPT.exists():
    fm_vit.load_state_dict(torch.load(FM_CKPT, map_location=DEVICE, weights_only=True))
    print(f"Loaded checkpoint: {FM_CKPT}")
    fm_history = None
else:
    fm_history = train(
        model=fm_vit, train_loader=fm_train_loader,
        val_loader=fm_test_loader, epochs=35, lr=3e-4,
        device=DEVICE, label="FashionMNIST ViT",
    )
    torch.save(fm_vit.state_dict(), FM_CKPT)
    print(f"\nCheckpoint saved → {FM_CKPT}")

final_fm_acc = evaluate(fm_vit, fm_test_loader, DEVICE)
print(f"Fashion-MNIST test accuracy: {final_fm_acc:.4f}  ({final_fm_acc*100:.2f}%)")

if fm_history:
    plot_history({"Fashion-MNIST ViT": fm_history}, title="ViT on Fashion-MNIST")

## 7. Training on CIFAR-10

CIFAR-10 images are **32 × 32 RGB** — three colour channels and a more complex, natural-image
distribution.  We scale up slightly:

| Change from MNIST config | Reason |
|---|---|
| `patch_size = 4` | Finer granularity on 32 px → gives 64 patches |
| `embed_dim = 256` | More capacity for colour / texture patterns |
| `num_heads = 8` | More heads to specialise on different features |
| `num_layers = 6` | Deeper model for harder task |
| Stronger augmentation | RandomCrop, ColorJitter, RandomErasing |

Training ViT from scratch on CIFAR-10 is notoriously tricky — ViT prefers more data.
We add **label smoothing**, **gradient clipping**, and a **linear warm-up** cosine LR schedule.
Typical from-scratch accuracy: **75–80 %** (compare: ResNet-18 ~93 %).

> This is exactly why pre-trained ViT (Section 9) is so valuable — pre-training on
> ImageNet-21k gives rich low-level representations that transfer immediately.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

cifar_train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.2)),
])
cifar_test_tf  = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

cifar_train = CIFAR10(DATA_DIR, train=True,  download=True, transform=cifar_train_tf)
cifar_test  = CIFAR10(DATA_DIR, train=False, download=True, transform=cifar_test_tf)

cifar_train_loader = DataLoader(cifar_train, batch_size=256, shuffle=True,  num_workers=0)
cifar_test_loader  = DataLoader(cifar_test,  batch_size=512, shuffle=False, num_workers=0)

CIFAR_CLASSES = ["airplane","automobile","bird","cat","deer",
                 "dog","frog","horse","ship","truck"]

# Visualise (un-normalise for display)
cifar_inv_norm = T.Normalize(
    mean=[-m/s for m, s in zip(CIFAR_MEAN, CIFAR_STD)],
    std=[1/s for s in CIFAR_STD])

_raw_cifar = CIFAR10(DATA_DIR, train=True, download=False,
                     transform=T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)]))
fig, axes  = plt.subplots(2, 10, figsize=(15, 3.5))
for i, ax in enumerate(axes.flatten()):
    img, label = _raw_cifar[i]
    ax.imshow(cifar_inv_norm(img).permute(1, 2, 0).clip(0, 1))
    ax.set_title(CIFAR_CLASSES[label], fontsize=7)
    ax.axis("off")
plt.suptitle("CIFAR-10 samples", y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
def train_cifar(model, train_loader, val_loader, epochs, lr_max, warmup_epochs, device):
    """CIFAR-10 trainer with linear warm-up, cosine decay, label smoothing, grad clip."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=5e-4)
    history   = defaultdict(list)

    steps_per_epoch = len(train_loader)
    warmup_steps    = warmup_epochs * steps_per_epoch
    total_steps     = epochs * steps_per_epoch

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * prog))

    scheduler   = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda s: lr_lambda(s) * lr_max / 1e-6)
    global_step = 0

    epoch_bar = tqdm(range(1, epochs + 1), desc="CIFAR-10 ViT", unit="epoch")
    for epoch in epoch_bar:
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        batch_bar = tqdm(train_loader, desc=f"  E{epoch:02d}", leave=False, unit="batch")
        for imgs, labels in batch_bar:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = F.cross_entropy(logits, labels, label_smoothing=0.1)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            global_step += 1
            total_loss += loss.item() * imgs.size(0)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
            batch_bar.set_postfix(loss=f"{loss.item():.3f}",
                                  lr=f"{scheduler.get_last_lr()[0]:.2e}")
        val_acc = evaluate(model, val_loader, device)
        history["train_loss"].append(total_loss / total)
        history["train_acc"].append(correct / total)
        history["val_acc"].append(val_acc)
        epoch_bar.set_postfix(loss=f"{total_loss/total:.4f}", val=f"{val_acc:.3f}")
    return history

In [ ]:
CIFAR_CKPT = VIT_DIR / "cifar10_vit.pth"

cifar_vit = VisionTransformer(
    image_size=32, patch_size=4, in_channels=3, num_classes=10,
    embed_dim=256, num_heads=8, num_layers=6,
    mlp_ratio=4.0, attn_drop=0.1, drop_rate=0.1,
).to(DEVICE)

if CIFAR_CKPT.exists():
    cifar_vit.load_state_dict(torch.load(CIFAR_CKPT, map_location=DEVICE, weights_only=True))
    print(f"Loaded checkpoint: {CIFAR_CKPT}")
    cifar_history = None
else:
    n = sum(p.numel() for p in cifar_vit.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n:,}")
    cifar_history = train_cifar(
        model=cifar_vit, train_loader=cifar_train_loader,
        val_loader=cifar_test_loader,
        epochs=30, lr_max=3e-4, warmup_epochs=5, device=DEVICE,
    )
    torch.save(cifar_vit.state_dict(), CIFAR_CKPT)
    print(f"\nCheckpoint saved → {CIFAR_CKPT}")

final_cifar_acc = evaluate(cifar_vit, cifar_test_loader, DEVICE)
print(f"CIFAR-10 test accuracy: {final_cifar_acc:.4f}  ({final_cifar_acc*100:.2f}%)")

if cifar_history:
    plot_history({"CIFAR-10 ViT": cifar_history}, title="ViT from Scratch on CIFAR-10")

## 8. Attention-Map Visualisation

### What the original paper shows

Figure 6 of *"An Image is Worth 16×16 Words"* overlays the attention weights of the `[CLS]` token
on the input image.  Concretely:

- After a forward pass, each head in the **last Transformer block** produces an attention matrix
  of shape `(N+1, N+1)`.
- The row corresponding to `[CLS]` (row 0) shows which patches the token attends to.
- We extract that row, drop the `[CLS]`→`[CLS]` entry, reshape `N → (H/P, W/P)`, and upsample
  back to the original image size.

### Attention Rollout

A more principled approach is **Attention Rollout** (Abnar & Zuidema, 2020):
instead of using only the last layer, we *roll* attention through all layers by multiplying
them together, accounting for residual connections.  This captures how information flows
from early patches all the way to the `[CLS]` token.

### How We Extract Attention Maps

Our custom ViT stores attention weights during every forward pass inside
`MultiHeadSelfAttention.last_attn_weights` (set in the `forward()` hook).
This lets us retrieve them without any re-computation.

#### Method A — Last-Layer CLS Attention

```
attn = model.get_attention_weights(-1)   # (1, H, N+1, N+1)
                                         # last block, all heads, full seq × seq matrix
attn = attn.mean(dim=1)                  # average over H heads → (1, N+1, N+1)
cls_row = attn[0, 0, 1:]                 # row 0 = CLS token, columns 1: = patches
```

We then reshape `cls_row` from a flat `(N,)` vector to a `(√N, √N)` grid and upsample.
This is the approach used in Figure 6 of the original ViT paper.

**Limitation:** using only the last layer ignores how information built up through earlier layers.

#### Method B — Attention Rollout (Abnar & Zuidema, 2020)

Attention rollout recursively multiplies attention matrices across layers,
accounting for residual connections (which route some information "around" attention):

```
A_eff[l] = 0.5 * A[l]  +  0.5 * I      (residual term)
rollout   = A_eff[L] @ A_eff[L-1] @ … @ A_eff[1]
```

The CLS row of `rollout` reflects how much each input patch contributed to the
final CLS token, folding all layers together.  It is smoother and often more faithful
than the single-layer view, especially in deep models.

**Colour map:** we use `inferno` (dark purple → yellow → white) because it has high
perceptual contrast and is colour-blind friendly.

In [ ]:
def cls_attention_map(model: VisionTransformer, img_tensor: Tensor,
                      layer: int = -1) -> np.ndarray:
    """Last-layer CLS attention map, upsampled to image resolution."""
    model.eval()
    with torch.no_grad():
        _ = model(img_tensor)
    attn = model.get_attention_weights(layer)   # (1, H, N+1, N+1)
    attn = attn.mean(dim=1)                     # average heads → (1, N+1, N+1)
    cls_attn = attn[0, 0, 1:]                   # CLS row, skip CLS→CLS  (N,)

    ph = int(cls_attn.shape[0] ** 0.5)
    amap = cls_attn.reshape(ph, ph).cpu().numpy()
    patch_size = img_tensor.shape[-1] // ph
    amap = np.kron(amap, np.ones((patch_size, patch_size)))
    amap = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)
    return amap


def attention_rollout(model: VisionTransformer, img_tensor: Tensor) -> np.ndarray:
    """Attention Rollout across all layers (Abnar & Zuidema, 2020)."""
    model.eval()
    with torch.no_grad():
        _ = model(img_tensor)

    rollout = None
    for block in model.blocks:
        attn = block.attn.last_attn_weights   # (1, H, N+1, N+1)
        attn = attn.mean(dim=1)[0]            # (N+1, N+1)
        I    = torch.eye(attn.size(0), device=attn.device)
        attn = 0.5 * attn + 0.5 * I
        attn = attn / attn.sum(dim=-1, keepdim=True)
        rollout = attn if rollout is None else attn @ rollout

    cls_attn = rollout[0, 1:]
    ph = int(cls_attn.shape[0] ** 0.5)
    amap = cls_attn.reshape(ph, ph).cpu().numpy()
    patch_size = img_tensor.shape[-1] // ph
    amap = np.kron(amap, np.ones((patch_size, patch_size)))
    amap = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)
    return amap


def show_attention(model, dataset, inv_transform, classes, n=6, cmap="inferno",
                   title_prefix="", device=DEVICE, is_grayscale=False):
    """Input | Last-layer CLS attention | Attention Rollout side by side."""
    fig, axes = plt.subplots(n, 3, figsize=(10, 2.8 * n))
    for ax, c in zip(axes[0], ["Input", "CLS Attention (last layer)", "Attention Rollout"]):
        ax.set_title(c, fontsize=11, fontweight="bold")

    idxs = np.random.choice(len(dataset), n, replace=False)
    for row, idx in enumerate(idxs):
        img_raw, label = dataset[idx]
        img_t = img_raw.unsqueeze(0).to(device)

        if is_grayscale:
            img_disp = img_raw.squeeze().numpy()
            axes[row, 0].imshow(img_disp, cmap="gray")
        else:
            img_disp = inv_transform(img_raw).permute(1, 2, 0).numpy().clip(0, 1)
            axes[row, 0].imshow(img_disp)
        axes[row, 0].set_ylabel(classes[label], rotation=0, labelpad=50,
                                va="center", fontsize=10)

        for col, amap in enumerate([cls_attention_map(model, img_t),
                                    attention_rollout(model, img_t)], start=1):
            axes[row, col].imshow(amap, cmap=cmap)
            if is_grayscale:
                axes[row, col].imshow(img_disp, cmap="gray", alpha=0.35)
            else:
                axes[row, col].imshow(img_disp, alpha=0.35)

        for ax in axes[row]:
            ax.axis("off")

    plt.suptitle(f"{title_prefix} — Attention Maps", fontsize=13, y=1.01)
    plt.tight_layout(); plt.show()

#### MNIST Attention Maps

In [ ]:
show_attention(
    model=mnist_vit,
    dataset=MNIST(DATA_DIR, train=False, download=False, transform=mnist_test_tf),
    inv_transform=lambda x: x,
    classes=[str(i) for i in range(10)],
    n=6, title_prefix="MNIST", is_grayscale=True,
)

#### Fashion-MNIST Attention Maps

In [ ]:
show_attention(
    model=fm_vit,
    dataset=FashionMNIST(DATA_DIR, train=False, download=False, transform=fm_test_tf),
    inv_transform=lambda x: x,
    classes=FM_CLASSES, n=6, title_prefix="Fashion-MNIST", is_grayscale=True,
)

#### CIFAR-10 Attention Maps

In [ ]:
show_attention(
    model=cifar_vit,
    dataset=CIFAR10(DATA_DIR, train=False, download=False, transform=cifar_test_tf),
    inv_transform=cifar_inv_norm,
    classes=CIFAR_CLASSES, n=6, title_prefix="CIFAR-10", is_grayscale=False,
)

#### Per-Head Comparison — all attention heads at once

In [ ]:
def show_all_heads(model, img_tensor, img_disp, label_str, is_grayscale=False):
    """Show each attention head of the last layer separately."""
    model.eval()
    with torch.no_grad():
        _ = model(img_tensor)
    attn = model.get_attention_weights(-1)   # (1, H, N+1, N+1)
    H    = attn.shape[1]
    fig, axes = plt.subplots(2, H // 2, figsize=(H * 1.8, 4))
    for h, ax in enumerate(axes.flatten()):
        head_attn = attn[0, h, 0, 1:].cpu().numpy()
        ph = int(head_attn.shape[0] ** 0.5)
        amap = np.kron(head_attn.reshape(ph, ph),
                       np.ones((img_tensor.shape[-1] // ph,) * 2))
        amap = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)
        ax.imshow(amap, cmap="inferno")
        if is_grayscale:
            ax.imshow(img_disp, cmap="gray", alpha=0.35)
        else:
            ax.imshow(img_disp, alpha=0.35)
        ax.set_title(f"Head {h}", fontsize=9); ax.axis("off")
    plt.suptitle(f"All heads — last layer | {label_str}", fontsize=12, y=1.02)
    plt.tight_layout(); plt.show()

_cifar_raw = CIFAR10(DATA_DIR, train=False, download=False, transform=cifar_test_tf)
_img_t, _label = _cifar_raw[7]
_disp = cifar_inv_norm(_img_t).permute(1, 2, 0).numpy().clip(0, 1)
show_all_heads(cifar_vit, _img_t.unsqueeze(0).to(DEVICE), _disp,
               label_str=CIFAR_CLASSES[_label])

## 9. Fine-tuning a Pre-trained ViT (HuggingFace)

### Why fine-tune instead of training from scratch?

The from-scratch CIFAR-10 result (~75–80 %) is significantly below CNN baselines (~93 %).
The reason is well understood: **ViT has no inductive biases** — no translation invariance,
no local convolutions — and must learn them from data alone.  With only 50k images it cannot.

**Solution:** use a model pre-trained on a much larger corpus.
We use `google/vit-base-patch16-224` — pre-trained on **ImageNet-21k** (~14 M images).

Fine-tuning strategy:
1. Replace the classification head with `nn.Linear(768, 10)`
2. **Freeze** all layers except the head + last 2 Transformer blocks (parameter-efficient)
3. Small LR (5e-5) + cosine schedule
4. Resize CIFAR-10 images 32 → 224 px (required by the 16×16 patch tokeniser)
5. Only **5 000** training images — 10× less than from-scratch yet higher accuracy

In [ ]:
from transformers import ViTForImageClassification, ViTImageProcessor

HF_MODEL_ID = "google/vit-base-patch16-224"
processor   = ViTImageProcessor.from_pretrained(HF_MODEL_ID)
hf_mean, hf_std = processor.image_mean, processor.image_std
img_sz = processor.size["height"]   # 224
print("Expected input size:", processor.size)

In [ ]:
hf_train_tf = T.Compose([
    T.Resize((img_sz, img_sz)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToTensor(),
    T.Normalize(hf_mean, hf_std),
])
hf_test_tf = T.Compose([
    T.Resize((img_sz, img_sz)),
    T.ToTensor(),
    T.Normalize(hf_mean, hf_std),
])

_full_train = CIFAR10(DATA_DIR, train=True,  download=False, transform=hf_train_tf)
_full_test  = CIFAR10(DATA_DIR, train=False, download=False, transform=hf_test_tf)

rng     = np.random.default_rng(42)
sub_idx = rng.choice(len(_full_train), size=5_000, replace=False)
hf_train_ds = Subset(_full_train, sub_idx)
hf_test_ds  = _full_test

hf_train_loader = DataLoader(hf_train_ds, batch_size=32, shuffle=True,  num_workers=0)
hf_test_loader  = DataLoader(hf_test_ds,  batch_size=64, shuffle=False, num_workers=0)
print(f"Fine-tune train: {len(hf_train_ds):,}  |  test: {len(hf_test_ds):,}")

In [ ]:
def freeze_for_finetuning(model, unfreeze_last_n_blocks: int = 2):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True
    total = len(model.vit.encoder.layer)
    for block in model.vit.encoder.layer[total - unfreeze_last_n_blocks:]:
        for p in block.parameters():
            p.requires_grad = True
    for p in model.vit.layernorm.parameters():
        p.requires_grad = True

HF_CKPT = VIT_DIR / "cifar10_vit_finetuned.pth"

hf_vit = ViTForImageClassification.from_pretrained(
    HF_MODEL_ID, num_labels=10, ignore_mismatched_sizes=True)
freeze_for_finetuning(hf_vit, unfreeze_last_n_blocks=2)

trainable = sum(p.numel() for p in hf_vit.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in hf_vit.parameters())
print(f"Trainable: {trainable:,} / {total_p:,}  ({100*trainable/total_p:.1f}%)")
hf_vit = hf_vit.to(DEVICE)

In [ ]:
def train_hf_vit(model, train_loader, test_loader, epochs, lr, device):
    """Fine-tune HuggingFace ViT with tqdm progress bars."""
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(train_loader))
    history   = defaultdict(list)

    epoch_bar = tqdm(range(1, epochs + 1), desc="Fine-tuning HF ViT", unit="epoch")
    for epoch in epoch_bar:
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        batch_bar = tqdm(train_loader, desc=f"  E{epoch:02d}", leave=False, unit="batch")
        for imgs, labels in batch_bar:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out  = model(pixel_values=imgs)
            loss = F.cross_entropy(out.logits, labels, label_smoothing=0.1)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item() * imgs.size(0)
            correct    += (out.logits.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
            batch_bar.set_postfix(loss=f"{loss.item():.3f}")

        model.eval()
        val_c, val_t = 0, 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                val_c += (model(pixel_values=imgs).logits.argmax(1) == labels).sum().item()
                val_t += imgs.size(0)

        tr_acc  = correct / total
        val_acc = val_c / val_t
        history["train_loss"].append(total_loss / total)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(val_acc)
        epoch_bar.set_postfix(loss=f"{total_loss/total:.4f}",
                              tr=f"{tr_acc:.3f}", val=f"{val_acc:.3f}")
    return history

In [ ]:
if HF_CKPT.exists():
    hf_vit.load_state_dict(torch.load(HF_CKPT, map_location=DEVICE, weights_only=True))
    print(f"Loaded checkpoint: {HF_CKPT}")
    hf_history = None
else:
    hf_history = train_hf_vit(
        model=hf_vit, train_loader=hf_train_loader, test_loader=hf_test_loader,
        epochs=10, lr=5e-5, device=DEVICE,
    )
    torch.save(hf_vit.state_dict(), HF_CKPT)
    print(f"\nCheckpoint saved → {HF_CKPT}")

hf_final = hf_history["val_acc"][-1] if hf_history else evaluate(
    lambda imgs: type("", (), {"logits": hf_vit(pixel_values=imgs).logits})(),
    hf_test_loader, DEVICE)

if hf_history:
    plot_history({"Pre-trained ViT (5k labels)": hf_history},
                 title="Fine-tuned ViT on CIFAR-10")

print("\n=== Accuracy Comparison ===")
print(f"  ViT from scratch (50k labels):  {final_cifar_acc:.3f}")
print(f"  Fine-tuned ViT  ( 5k labels):  {hf_history['val_acc'][-1] if hf_history else 'loaded'}")

#### Attention maps from the HuggingFace pre-trained ViT

In [ ]:
# Inverse-normalise for display (undo HuggingFace preprocessing mean/std)
hf_inv = T.Normalize(mean=[-m/s for m,s in zip(hf_mean, hf_std)],
                     std=[1/s for s in hf_std])


def get_hf_attention_map(model, img_tensor: torch.Tensor) -> np.ndarray:
    """
    Extract the CLS-token attention map from the last encoder layer of a
    HuggingFace ViTForImageClassification model.

    Why forward hooks instead of output_attentions=True?
    ─────────────────────────────────────────────────────
    In transformers ≥ 5.x the ViT internals were refactored:
      • ViTAttention.forward discards probs:  `self_attn_output, _ = self.attention(...)`
      • ViTEncoder.forward no longer collects per-layer attentions
      • SDPA backend (default) returns None for attention weights

    The reliable solution: hook the query and key Linear projections inside
    the last ViTSelfAttention block, capture their outputs, and recompute the
    attention matrix manually using the standard scaled dot-product formula.
    This works with every transformers version and attention implementation.

    Returns
    ───────
    amap : np.ndarray  shape (H, W), values in [0, 1]
        Attention heat-map aligned with the input image dimensions.
    """
    model.eval()
    q_buf: dict = {}
    k_buf: dict = {}

    # Target: the ViTSelfAttention inside the last encoder block
    # model.vit.encoder.layer[-1]          → ViTLayer
    #   .attention                          → ViTAttention  (wrapper + output proj)
    #     .attention                        → ViTSelfAttention  (Q K V + scores)
    sa = model.vit.encoder.layer[-1].attention.attention

    # Register hooks on the Q and K linear layers to capture their outputs
    # Output shapes:  (batch, seq_len, all_head_size)  =  (1, 197, 768)
    h_q = sa.query.register_forward_hook(
        lambda m, inp, out: q_buf.update({"v": out.detach()}))
    h_k = sa.key.register_forward_hook(
        lambda m, inp, out: k_buf.update({"v": out.detach()}))

    with torch.no_grad():
        _ = model(pixel_values=img_tensor)   # single forward pass

    # Always remove hooks — even if the forward raises
    h_q.remove()
    h_k.remove()

    # ── Recompute multi-head attention from Q and K ────────────────────────────
    n_heads  = sa.num_attention_heads    # 12  for ViT-Base
    head_dim = sa.attention_head_size    # 64  for ViT-Base  (768 / 12)
    scaling  = head_dim ** -0.5         # 1/sqrt(64) ≈ 0.125

    def split_heads(x: torch.Tensor) -> torch.Tensor:
        """(B, N, all_head_size) → (B, n_heads, N, head_dim)"""
        B, N, _ = x.shape
        return x.view(B, N, n_heads, head_dim).transpose(1, 2)

    q = split_heads(q_buf["v"])          # (1, 12, 197, 64)
    k = split_heads(k_buf["v"])          # (1, 12, 197, 64)

    attn = (q @ k.transpose(-2, -1)) * scaling   # (1, 12, 197, 197)
    attn = attn.softmax(dim=-1)

    # ── Extract CLS-row attention ──────────────────────────────────────────────
    # Average over the 12 heads, then take row 0 (= CLS token),
    # skip column 0 (= CLS→CLS self-attention), keep columns 1: (patch tokens)
    cls_attn = attn[0].mean(0)[0, 1:]            # (196,)

    # Reshape to 2-D patch grid and upsample to image resolution
    ph   = int(cls_attn.shape[0] ** 0.5)         # 14  (14×14 = 196 patches)
    amap = cls_attn.reshape(ph, ph).cpu().numpy()
    amap = np.kron(amap, np.ones((img_tensor.shape[-1] // ph,) * 2))  # → (224, 224)

    return (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)


# ── Visualise on 3 CIFAR-10 test images (upscaled to 224px) ───────────────────
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
col_titles = ["Input (224px)", "CLS Attention — last layer", "Attention Overlay"]
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=10, fontweight="bold")

for row in range(3):
    img_t, label = _full_test[row * 300]
    img_disp = hf_inv(img_t).permute(1, 2, 0).numpy().clip(0, 1)
    amap = get_hf_attention_map(hf_vit, img_t.unsqueeze(0).to(DEVICE))

    axes[row, 0].imshow(img_disp)
    axes[row, 0].set_title(CIFAR_CLASSES[label], fontsize=9)
    axes[row, 1].imshow(amap, cmap="inferno")
    axes[row, 2].imshow(img_disp)
    axes[row, 2].imshow(amap, cmap="inferno", alpha=0.55)
    for ax in axes[row]:
        ax.axis("off")

plt.suptitle(
    "Pre-trained ViT-Base/16 attention (CIFAR-10 images upscaled to 224px)\n"
    "Bright = high attention from CLS token. "
    "Focus shifts to the object after ImageNet-21k pre-training.",
    fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## 10. Zero-Shot Classification with OpenCLIP

### What is CLIP?

**CLIP** (Contrastive Language–Image Pre-Training, Radford et al. 2021) jointly trains a
**vision encoder** (ViT) and a **text encoder** (Transformer) to map images and captions
close together in a shared embedding space.

**Zero-shot classification** with CLIP:
1. Embed the image → vector `v_img`
2. For each class, embed the text prompt `"a photo of a {class}"` → `v_text_k`
3. Classify as `argmax_k  cosine_sim(v_img, v_text_k)`

**OpenCLIP** is an open-source reproduction trained on the public **LAION-2B** dataset
(2 billion image–text pairs), achieving results comparable to or better than OpenAI CLIP.

In [ ]:
import open_clip

CLIP_MODEL_NAME = "ViT-B-32"
CLIP_PRETRAINED  = "laion2b_s34b_b79k"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED)
clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)
clip_model = clip_model.to(DEVICE).eval()
print(f"OpenCLIP {CLIP_MODEL_NAME} loaded | params: {sum(p.numel() for p in clip_model.parameters()):,}")

In [ ]:
PROMPT_TEMPLATES = [
    "a photo of a {}.",
    "a high-resolution photo of a {}.",
    "a blurry photo of a {}.",
    "a photograph of a {}.",
    "a picture of a {}.",
]

@torch.no_grad()
def embed_text_prompts(model, tokenizer, classes, templates, device):
    embeddings = []
    for cls in classes:
        texts = tokenizer([t.format(cls) for t in templates]).to(device)
        embs  = model.encode_text(texts)
        embs  = embs / embs.norm(dim=-1, keepdim=True)
        embeddings.append(embs.mean(0))
    embeddings = torch.stack(embeddings)
    return embeddings / embeddings.norm(dim=-1, keepdim=True)

text_embs = embed_text_prompts(clip_model, clip_tokenizer,
                                CIFAR_CLASSES, PROMPT_TEMPLATES, DEVICE)
print("Text embedding matrix:", text_embs.shape)

In [ ]:
@torch.no_grad()
def zero_shot_accuracy(model, text_embs, preprocess, device):
    clip_ds = CIFAR10(DATA_DIR, train=False, download=False, transform=preprocess)
    loader  = DataLoader(clip_ds, batch_size=128, shuffle=False, num_workers=0)
    correct, total = 0, 0
    for imgs, labels in tqdm(loader, desc="Zero-shot eval", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        img_embs = model.encode_image(imgs)
        img_embs = img_embs / img_embs.norm(dim=-1, keepdim=True)
        preds    = (img_embs @ text_embs.T * model.logit_scale.exp()).argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
    return correct / total

clip_acc = zero_shot_accuracy(clip_model, text_embs, clip_preprocess, DEVICE)
print(f"OpenCLIP zero-shot CIFAR-10 accuracy: {clip_acc:.4f}  ({clip_acc*100:.2f}%)")

In [ ]:
# ── Prompt Engineering ────────────────────────────────────────────────────────
# CLIP is sensitive to how you phrase class names.
# Here we compare four strategies: a bare label, a 'photo of a' prefix,
# a more descriptive sentence, and an ensemble of 5 different phrasings.
#
# Key rule: every template must contain exactly one {} placeholder (or use {0}
# to repeat the same arg twice). Mixing two {} with one arg → IndexError.

prompt_variants = {
    "Simple":       ["{}."],
    "Photo":        ["a photo of a {}."],
    # {0} repeats the single class argument in both positions
    "Descriptive":  ["a high-quality photo of a {0}, a type of {0}."],
    "Ensemble (5)": PROMPT_TEMPLATES,   # 5 templates averaged → best accuracy
}
print("Prompt engineering effect on CIFAR-10 zero-shot accuracy:")
print("-" * 56)
for name, templates in prompt_variants.items():
    temb = embed_text_prompts(clip_model, clip_tokenizer, CIFAR_CLASSES, templates, DEVICE)
    acc  = zero_shot_accuracy(clip_model, temb, clip_preprocess, DEVICE)
    bar  = "█" * int(acc * 40)
    print(f"  {name:<16}: {acc:.3f}  {bar}")

print()
print("Takeaway: ensembling multiple prompt phrasings consistently outperforms")
print("any single template because it reduces sensitivity to wording quirks.")

In [ ]:
# Text-text similarity matrix
sim = (text_embs @ text_embs.T).cpu().numpy()
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(10)); ax.set_xticklabels(CIFAR_CLASSES, rotation=45, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(CIFAR_CLASSES)
for i in range(10):
    for j in range(10):
        ax.text(j, i, f"{sim[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if sim[i,j] > 0.7 else "black")
plt.colorbar(im, ax=ax, label="cosine similarity")
ax.set_title("CLIP text embedding similarity — CIFAR-10 classes", fontsize=12)
plt.tight_layout(); plt.show()

## 11. TIPSv2 & EUPE — Large-Scale ViT Feature Visualisation

In this section we load two state-of-the-art ViT encoders trained at massive scale and
visualise their internal patch representations.  This illustrates how much richer the
features become with scale, compared to the ViTs we trained from scratch.

| Model | Source | Pre-training data | Architecture |
|---|---|---|---|
| **TIPSv2-B/14** | Google (CVPR 2026) | Large-scale image-text pairs | ViT-B / patch 14 |
| **EUPE-ViT-B** | Meta AI | Multi-expert distillation | ViT-B / patch 16 |

We will visualise two complementary things:
1. **CLS-patch cosine similarity** — which patches are most similar to the global CLS token
   (analogous to attention maps but derived from feature similarity)
2. **PCA feature maps** — project patch features to 3 principal components and display as RGB;
   this reveals semantic segmentation structure learned purely from pre-training

### 11.1 Google TIPSv2

**TIPS** = *Text-Image Pre-training with Spatial awareness* (Zhai et al., CVPR 2026).
TIPSv2 improves patch-level text alignment — unlike CLIP which only aligns the CLS token,
TIPS also aligns individual patch tokens with textual descriptions of image regions.
This gives it much stronger spatial / segmentation features.

Key specs for `google/tipsv2-b14`:
- Vision encoder: ViT-B with **patch size 14** → 32×32 = 1024 patch tokens for 448px input
- Embedding dim: **768**
- **No ImageNet normalization** — images must be in `[0, 1]` range only
- Loaded via `AutoModel` with `trust_remote_code=True`
- License: Apache 2.0

In [ ]:
# sentencepiece is required for TIPSv2's text encoder tokeniser.
# Install it now if the import would fail.
try:
    import sentencepiece  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, "uv", "add", "sentencepiece"], check=True)
    import sentencepiece  # noqa: F401

from transformers import AutoModel

TIPS_MODEL_ID = "google/tipsv2-b14"
print(f"Loading {TIPS_MODEL_ID} …")
# trust_remote_code=True is required: TIPSv2 ships custom modelling code
# (image_encoder.py, text_encoder.py, modeling_tips.py) that is not yet
# part of the transformers library proper.
tips_model = AutoModel.from_pretrained(TIPS_MODEL_ID, trust_remote_code=True)
tips_model = tips_model.to(DEVICE).eval()
print(f"TIPSv2-B/14 loaded  |  vision: ViT-B/14, 86M params, embed=768")

# ── TIPSv2 image preprocessing ────────────────────────────────────────────────
# IMPORTANT: TIPSv2 does NOT use ImageNet normalisation.
# Images must be in the range [0, 1] (ToTensor only).
# The model was trained on 448×448 crops; use that size for best results.
TIPS_SIZE = 448                    # 448 / 14 (patch size) = 32 patches per side
tips_tf = T.Compose([
    T.Resize((TIPS_SIZE, TIPS_SIZE)),
    T.ToTensor(),                  # uint8 [0,255] → float32 [0,1]
                                   # ← deliberately NO Normalize() here
])

In [ ]:
@torch.no_grad()
def tips_patch_features(model, pil_img: Image.Image):
    """Return (cls_token, patch_tokens) for a PIL image."""
    img_t = tips_tf(pil_img).unsqueeze(0).to(next(model.parameters()).device)
    out   = model.encode_image(img_t)
    # out.cls_token:   (1, 1, 768)
    # out.patch_tokens:(1, N, 768)  — N = (448/14)^2 = 1024
    cls     = out.cls_token[:, 0, :]                          # (1, 768)
    patches = out.patch_tokens                                 # (1, 1024, 768)
    return cls, patches


def cls_patch_similarity_map(cls: Tensor, patches: Tensor,
                             grid_h: int, grid_w: int,
                             target_h: int, target_w: int) -> np.ndarray:
    """
    Compute cosine similarity of cls to each patch token, reshape to grid,
    bilinearly upsample to (target_h, target_w).
    """
    import torch.nn.functional as F
    cls_n     = F.normalize(cls, dim=-1)           # (1, D)
    patch_n   = F.normalize(patches, dim=-1)       # (1, N, D)
    sim       = (cls_n.unsqueeze(1) * patch_n).sum(-1).squeeze(0)  # (N,)
    sim_map   = sim.reshape(grid_h, grid_w).cpu().numpy()
    # Bilinear upsample via PIL
    pil_map   = Image.fromarray(
        ((sim_map - sim_map.min()) / (sim_map.max() - sim_map.min() + 1e-8) * 255
         ).astype(np.uint8)).resize((target_w, target_h), Image.BILINEAR)
    result    = np.array(pil_map, dtype=np.float32) / 255.0
    return result


def pca_feature_map(patches: Tensor, grid_h: int, grid_w: int,
                    target_h: int, target_w: int) -> np.ndarray:
    """
    Project patch tokens to top-3 PCA components → RGB image.
    Reveals semantic segmentation structure.
    """
    from sklearn.decomposition import PCA
    feat = patches.squeeze(0).cpu().float().numpy()      # (N, D)
    pca  = PCA(n_components=3)
    proj = pca.fit_transform(feat)                       # (N, 3)
    proj = (proj - proj.min(0)) / (proj.max(0) - proj.min(0) + 1e-8)
    rgb  = proj.reshape(grid_h, grid_w, 3)
    pil_rgb = Image.fromarray((rgb * 255).astype(np.uint8)).resize(
        (target_w, target_h), Image.BILINEAR)
    return np.array(pil_rgb) / 255.0

#### How to Read TIPSv2 Patch Feature Maps

We visualise TIPSv2's internal representations using two complementary techniques:

**1. CLS-to-Patch Cosine Similarity (middle column)**

After a forward pass, TIPSv2 returns:
- `cls_token` — a single 768-dim vector summarising the whole image (like BERT's `[CLS]`)
- `patch_tokens` — 1024 × 768 spatial feature vectors (one per 14×14 patch of the 448px image)

We compute the cosine similarity between the CLS vector and every patch vector:

```
sim[i] = cos_sim(cls_token, patch_tokens[i])     i ∈ {0 … 1023}
```

Then reshape the 1024 similarities to a 32×32 grid and upsample to display size.
**Bright = the patch is highly similar to the global image summary.**
This is semantically analogous to a self-attention map, but derived from feature similarity
rather than learned attention weights.

**2. PCA RGB Feature Map (right column)**

We take all 1024 patch feature vectors (shape 1024 × 768), fit PCA and project to 3 dimensions.
Each dimension is mapped to R, G, or B after min-max scaling:

```
patch_rgb[i] = PCA(patch_tokens[i])[:3]   scaled to [0, 1]
```

Patches that occupy the same colour region belong to the same cluster in feature space —
which often corresponds to the same semantic region (background sky, object body, ground).
This emergent segmentation arises from the model's text-image patch-alignment objective,
not from any segmentation supervision.

In [ ]:
# Collect some CIFAR-10 test PIL images (6 samples)
_pil_cifar_raw = CIFAR10(DATA_DIR, train=False, download=False)
_sample_idxs   = [0, 50, 200, 350, 700, 999]
_pil_samples   = [(idx, *_pil_cifar_raw[idx]) for idx in _sample_idxs]
# each element: (idx, pil_image, label)

TIPS_GRID = 32          # 448 / 14 = 32 patches per side
DISP_SIZE = 224         # display at 224px

fig, axes = plt.subplots(len(_pil_samples), 3,
                         figsize=(9, 3 * len(_pil_samples)))
axes[0, 0].set_title("Input", fontsize=11, fontweight="bold")
axes[0, 1].set_title("CLS-Patch Similarity", fontsize=11, fontweight="bold")
axes[0, 2].set_title("PCA of Patch Features (RGB)", fontsize=11, fontweight="bold")

for row, (idx, pil_img, label) in enumerate(_pil_samples):
    pil_disp = pil_img.resize((DISP_SIZE, DISP_SIZE))
    cls, patches = tips_patch_features(tips_model, pil_img)

    sim_map = cls_patch_similarity_map(cls, patches,
                                       TIPS_GRID, TIPS_GRID, DISP_SIZE, DISP_SIZE)
    pca_map = pca_feature_map(patches, TIPS_GRID, TIPS_GRID, DISP_SIZE, DISP_SIZE)

    axes[row, 0].imshow(pil_disp)
    axes[row, 0].set_ylabel(CIFAR_CLASSES[label], rotation=0, labelpad=50,
                             va="center", fontsize=10)
    axes[row, 1].imshow(sim_map, cmap="inferno")
    axes[row, 1].imshow(pil_disp, alpha=0.35)
    axes[row, 2].imshow(pca_map)
    for ax in axes[row]:
        ax.axis("off")

plt.suptitle("TIPSv2-B/14 — Patch Feature Visualisation (CIFAR-10)", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

**Reading the PCA map:** each colour represents a different patch-feature cluster.
Regions of the same colour share similar high-level semantics.
Notice how the model segments foreground objects from background even without any
segmentation supervision — this spatial awareness is the core contribution of TIPS.

For natural images at their native resolution the effect is even more dramatic.
Let's fetch one from the web to demonstrate:

In [ ]:
import requests
from io import BytesIO

# A random CC0 photo (cat)
_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg"
try:
    _resp = requests.get(_url, timeout=10)
    _natural_img = Image.open(BytesIO(_resp.content)).convert("RGB")
    cls_n, patches_n = tips_patch_features(tips_model, _natural_img)

    _disp_n = _natural_img.resize((DISP_SIZE, DISP_SIZE))
    _sim_n  = cls_patch_similarity_map(cls_n, patches_n,
                                        TIPS_GRID, TIPS_GRID, DISP_SIZE, DISP_SIZE)
    _pca_n  = pca_feature_map(patches_n, TIPS_GRID, TIPS_GRID, DISP_SIZE, DISP_SIZE)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(_disp_n); axes[0].set_title("Input")
    axes[1].imshow(_sim_n, cmap="inferno")
    axes[1].imshow(_disp_n, alpha=0.4); axes[1].set_title("CLS Similarity")
    axes[2].imshow(_pca_n); axes[2].set_title("PCA RGB features")
    for ax in axes: ax.axis("off")
    plt.suptitle("TIPSv2-B/14 on a natural image", fontsize=12)
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f"Could not fetch image ({e}). Run with an internet connection or provide a local PIL image.")

### 11.2 Meta EUPE — Efficient Universal Perception Encoder

**EUPE** = *Efficient Universal Perception Encoder* (Meta AI Research).
EUPE distils knowledge from multiple domain-expert foundation models
(depth, segmentation, detection, 3D) into a single compact ViT backbone.
The result is a model that provides strong features for all these tasks
simultaneously without any task-specific fine-tuning.

Key specs for `facebook/EUPE-ViT-B`:
- Architecture: **ViT-B/16** — 86M params, 16×16 patches
- Input: 224×224, standard ImageNet normalization
- Output: 1 CLS token + 196 patch tokens (14×14 grid)
- License: **FAIR Non-Commercial Research License**

⚠️ **Setup required:** EUPE uses a custom `torch.hub` loader that needs the EUPE repository.
The cell below clones it automatically if not present.  Internet access required once.

In [ ]:
from huggingface_hub import hf_hub_download

EUPE_REPO_DIR = Path("eupe_repo")
EUPE_HF_ID    = "facebook/EUPE-ViT-B"

# ── Step 1: clone the EUPE GitHub repo (needed for torch.hub loader) ──────────
# torch.hub.load with source='local' reads hubconf.py from the cloned directory.
# We use --depth 1 to download only the latest commit (saves time and space).
if not EUPE_REPO_DIR.exists():
    print("Cloning EUPE repository (one-time setup, ~10 MB) …")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/facebookresearch/eupe", str(EUPE_REPO_DIR)],
        check=True)
    print("Repository cloned.")
else:
    print(f"EUPE repo already present at {EUPE_REPO_DIR}/")

# ── Step 2: download the model checkpoint from HuggingFace Hub ────────────────
# The checkpoint in the facebook/EUPE-ViT-B repo is named 'EUPE-ViT-B.pt' (~400 MB).
# hf_hub_download caches it in ~/.cache/huggingface/hub so subsequent runs are instant.
print("Downloading EUPE-ViT-B checkpoint from HuggingFace Hub (~400 MB, cached after first run) …")
eupe_ckpt = hf_hub_download(repo_id=EUPE_HF_ID, filename="EUPE-ViT-B.pt")
print(f"Checkpoint ready at: {eupe_ckpt}")

In [ ]:
# ── Step 3: load the model via torch.hub ──────────────────────────────────────
# hubconf.py in the EUPE repo exposes 'eupe_vitb16' (and variants vits16, vitt16).
# source='local' tells torch.hub to use our cloned directory rather than GitHub.
# The 'weights' argument is forwarded to the model builder which loads the .pt file.
eupe_model = torch.hub.load(
    str(EUPE_REPO_DIR),     # path to cloned facebookresearch/eupe repo
    "eupe_vitb16",          # entry in hubconf.py (ViT-Base / patch-16)
    source="local",
    weights=eupe_ckpt,      # path to downloaded EUPE-ViT-B.pt
)
eupe_model = eupe_model.to(DEVICE).eval()
print(f"EUPE-ViT-B loaded  |  ViT-B/16, 86M params")
print(f"  CLS token dim  : 768")
print(f"  Patch grid     : 14×14 = 196 tokens  (for 224px input)")

# ── EUPE preprocessing ────────────────────────────────────────────────────────
# EUPE uses standard ImageNet normalisation — same as most torchvision models.
# The recommended inference size is 256px centre-cropped to 224px.
EUPE_MEAN = (0.485, 0.456, 0.406)
EUPE_STD  = (0.229, 0.224, 0.225)

eupe_tf = T.Compose([
    T.Resize(256),           # resize shortest edge to 256
    T.CenterCrop(224),       # centre crop to 224×224
    T.ToTensor(),            # [0,255] → [0,1]
    T.Normalize(EUPE_MEAN, EUPE_STD),
])
# Inverse transform for display (undo normalisation)
eupe_inv_norm = T.Normalize(
    mean=[-m/s for m, s in zip(EUPE_MEAN, EUPE_STD)],
    std=[1/s for s in EUPE_STD])

#### Visualising EUPE Patch Features

We apply the same two-map visualisation as for TIPSv2:

1. **CLS↔Patch cosine similarity** — which patches are most aligned with the
   global CLS token?  Bright patches contribute most to the overall image
   representation.

2. **PCA RGB map** — we project the 196 patch embeddings (each 768-dim) to their
   top 3 principal components and map them to R, G, B channels.  Patches that share
   the same colour belong to the same "semantic cluster" in feature space.
   This reveals that EUPE — despite being trained without explicit segmentation labels —
   implicitly learns to separate foreground objects from background.

Because EUPE's pre-training distils from multiple expert models (depth, surface normals,
semantic segmentation, object detection), its patch features are richer and more
interpretable than those of models trained only on classification.

> **Note:** CIFAR-10 images are only 32×32 pixels.  We upscale them to 224px before
> feeding them to EUPE (whose patch size is 16px → needs at least 16px per patch).
> The coarse input means some spatial detail is lost, but the feature semantics remain
> surprisingly meaningful.

In [ ]:
@torch.no_grad()
def eupe_patch_features(model, pil_img: Image.Image):
    """
    Run EUPE forward_features on a PIL image and return CLS + patch tokens.

    EUPE exposes two normalised outputs via forward_features():
      x_norm_clstoken   : (1, D)    — global image representation
      x_norm_patchtokens: (1, N, D) — per-patch spatial features  (N=196 for 224px)

    These have already been L2-normalised by the model's final LayerNorm,
    making cosine similarity directly comparable across images.
    """
    # Preprocess: resize + centre-crop + ImageNet normalisation
    img_t = eupe_tf(pil_img).unsqueeze(0).to(next(model.parameters()).device)
    # forward_features returns a dict (not a tuple) — EUPE's custom API
    out     = model.forward_features(img_t)
    cls     = out["x_norm_clstoken"]       # (1, 768)
    patches = out["x_norm_patchtokens"]    # (1, 196, 768)
    return cls, patches


EUPE_GRID = 14    # 224px / 16px patch = 14 patches per side → 14×14 = 196 tokens

# ── Visualisation: CLS-patch similarity + PCA feature map ─────────────────────
fig, axes = plt.subplots(len(_pil_samples), 3,
                         figsize=(9, 3 * len(_pil_samples)))
axes[0, 0].set_title("Input (224px)", fontsize=10, fontweight="bold")
axes[0, 1].set_title("CLS↔Patch Cosine Similarity", fontsize=10, fontweight="bold")
axes[0, 2].set_title("PCA of Patch Features (RGB)", fontsize=10, fontweight="bold")

for row, (idx, pil_img, label) in enumerate(_pil_samples):
    pil_disp = pil_img.resize((DISP_SIZE, DISP_SIZE))
    cls, patches = eupe_patch_features(eupe_model, pil_img)

    # Similarity map: which patches are most similar to the global CLS token?
    sim_map = cls_patch_similarity_map(cls, patches,
                                       EUPE_GRID, EUPE_GRID, DISP_SIZE, DISP_SIZE)
    # PCA map: project 768-dim patch features to 3D → display as RGB colour
    pca_map = pca_feature_map(patches, EUPE_GRID, EUPE_GRID, DISP_SIZE, DISP_SIZE)

    axes[row, 0].imshow(pil_disp)
    axes[row, 0].set_ylabel(CIFAR_CLASSES[label], rotation=0, labelpad=50,
                             va="center", fontsize=10)
    axes[row, 1].imshow(sim_map, cmap="inferno")
    axes[row, 1].imshow(pil_disp, alpha=0.3)   # light overlay for context
    axes[row, 2].imshow(pca_map)
    for ax in axes[row]:
        ax.axis("off")

plt.suptitle(
    "EUPE-ViT-B — Patch Feature Visualisation (CIFAR-10 images, 32px → 224px)\n"
    "EUPE learns multi-task features by distilling from depth, segmentation "
    "and detection experts.",
    fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

### 11.3 Side-by-side Model Comparison

Let us now compare all three ViT flavours on the same images:
- Our **CIFAR-10 scratch ViT** (last-layer CLS attention)
- **TIPSv2-B/14** (CLS-patch similarity)
- **EUPE-ViT-B** (CLS-patch similarity)

Notice how the representations evolve with scale and pre-training sophistication.

### 11.3 Side-by-Side Model Comparison

Now we run all three ViT flavours on the same CIFAR-10 test images and put their
"what does the CLS token look at?" maps side by side.

| Column | Model | How the map is computed |
|---|---|---|
| Scratch ViT | Our CIFAR-10 model | Attention weights from last encoder block, averaged over heads |
| TIPSv2-B/14 | Google (CVPR 2026) | Cosine similarity of CLS token to each of 1024 patch tokens (448px) |
| EUPE-ViT-B | Meta AI | Cosine similarity of CLS token to each of 196 patch tokens (224px) |

**What to look for:**
- The scratch ViT has coarser maps (8×8 = 64 patches) and may be more diffuse because it
  learned from only 50k CIFAR images.
- TIPSv2 has the finest grid (32×32 = 1024 patches) and should highlight foreground objects
  very precisely — this is the "spatial awareness" that TIPS explicitly optimises.
- EUPE has 14×14 = 196 patches and typically focuses on the most task-discriminative region
  because it was trained with signals from multiple expert models (depth, segmentation, …).

Even when the maps look different in absolute scale, notice that *all* models tend to put
higher weights on the object of interest rather than the background — that is the key insight
of ViT: self-attention provides global, content-based feature selection from the very first
layer, without any convolutional locality constraints.

In [ ]:
def scratch_attention_map_np(model, pil_img: Image.Image,
                              transform, device) -> np.ndarray:
    """
    Apply our from-scratch CIFAR-10 ViT to a PIL image and return the
    CLS-token attention map (last layer, heads averaged) as a numpy array
    already upsampled to (DISP_SIZE × DISP_SIZE).

    The transform argument must match what the model was trained with so that
    the patch embedding sees the correct scale / normalisation.
    """
    img_t = transform(pil_img).unsqueeze(0).to(device)
    return cls_attention_map(model, img_t, layer=-1)   # defined in Section 8


# ── Side-by-side comparison: Scratch ViT | TIPSv2 | EUPE ──────────────────────
fig_cols = [
    "Input", 
    "Scratch ViT (CLS attn, last layer)", 
    "TIPSv2-B/14(CLS↔patch cosine sim)", 
    "EUPE-ViT-B(CLS↔patch cosine sim)",
]
compare_idxs = [0, 200, 500, 800]

fig, axes = plt.subplots(len(compare_idxs), 4, figsize=(14, 4 * len(compare_idxs)))
for ax, title in zip(axes[0], fig_cols):
    ax.set_title(title, fontsize=9, fontweight="bold")

for row, idx in enumerate(compare_idxs):
    # _pil_cifar_raw is a CIFAR10 dataset with no transform (returns PIL images)
    pil_img, label = _pil_cifar_raw[idx]
    pil_disp = pil_img.resize((DISP_SIZE, DISP_SIZE))

    # ── Column 0: input image ─────────────────────────────────────────────────
    axes[row, 0].imshow(pil_disp)
    axes[row, 0].set_ylabel(CIFAR_CLASSES[label], rotation=0, labelpad=50,
                             va="center", fontsize=10)

    # ── Column 1: our scratch ViT attention ───────────────────────────────────
    # cifar_test_tf normalises the image to match training distribution
    scratch_map = scratch_attention_map_np(
        cifar_vit, pil_img, cifar_test_tf, DEVICE)
    axes[row, 1].imshow(scratch_map, cmap="inferno")
    axes[row, 1].imshow(pil_disp, alpha=0.35)

    # ── Column 2: TIPSv2 CLS-patch similarity ─────────────────────────────────
    cls_t, patches_t = tips_patch_features(tips_model, pil_img)
    tips_map = cls_patch_similarity_map(
        cls_t, patches_t, TIPS_GRID, TIPS_GRID, DISP_SIZE, DISP_SIZE)
    axes[row, 2].imshow(tips_map, cmap="inferno")
    axes[row, 2].imshow(pil_disp, alpha=0.35)

    # ── Column 3: EUPE CLS-patch similarity ───────────────────────────────────
    cls_e, patches_e = eupe_patch_features(eupe_model, pil_img)
    eupe_map = cls_patch_similarity_map(
        cls_e, patches_e, EUPE_GRID, EUPE_GRID, DISP_SIZE, DISP_SIZE)
    axes[row, 3].imshow(eupe_map, cmap="inferno")
    axes[row, 3].imshow(pil_disp, alpha=0.35)

    for ax in axes[row]:
        ax.axis("off")

plt.suptitle(
    "Model comparison: CLS feature similarity maps on CIFAR-10\n"
    "Scratch ViT trained on 50k images vs. models pre-trained on billions of image-text pairs.\n"
    "Brighter = more similar to the global CLS token representation.",
    fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

### Final Summary

| Method | Dataset | # Labels | Test Accuracy | Notes |
|---|---|---|---|---|
| ViT from scratch | MNIST | 60k | ~98 % | tiny model, 4 layers |
| ViT from scratch | Fashion-MNIST | 60k | ~90 % | same architecture |
| ViT from scratch | CIFAR-10 | 50k | ~75–80 % | limited by data |
| Pre-trained ViT (HF) | CIFAR-10 | 5k | ~90–93 % | ImageNet-21k init |
| OpenCLIP zero-shot | CIFAR-10 | **0** | ~76–80 % | language supervision only |

Key takeaways:
1. **ViT needs data** — from-scratch training on small datasets is hard without inductive biases.
2. **Pre-training is powerful** — 5k labels + ImageNet-21k pre-training beats 50k labels from scratch.
3. **Zero-shot CLIP** matches from-scratch ViT with *no labels at all*, using only natural language.
4. **TIPSv2** produces rich spatial patch features that capture semantic regions without supervision.
5. **EUPE** provides multi-task features by distilling from domain experts into one ViT backbone.
6. All weights are saved in `data/vit/` — re-run any section without retraining.

### Further Reading

- [An Image is Worth 16x16 Words (ViT)](https://arxiv.org/abs/2010.11929) — Dosovitskiy et al., 2020
- [Learning Transferable Visual Models from Natural Language (CLIP)](https://arxiv.org/abs/2103.00020) — Radford et al., 2021
- [Attention Rollout](https://arxiv.org/abs/2005.00928) — Abnar & Zuidema, 2020
- [TIPSv2: Advancing Vision-Language Pretraining](https://arxiv.org/abs/2604.12012) — Zhai et al., CVPR 2026
- [EUPE: Efficient Universal Perception Encoder](https://huggingface.co/facebook/EUPE-ViT-B) — Meta AI
- [OpenCLIP](https://github.com/mlfoundations/open_clip) — open-source CLIP
- [DINOv2](https://arxiv.org/abs/2304.07193) — Meta's self-supervised ViT